In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import sklearn 

I have decided to work on the hourly dataset : 

In [2]:
df=pd.read_csv('/home/yazid/sklearn/bikes/data/hour.csv')

In [3]:
df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [4]:
len(df)

17379

In [5]:
df.drop('instant',axis=1,inplace=True)

In [6]:
df['dteday']=pd.to_datetime(df['dteday'])

Let's create some features :

Cyclical temporical features :

In [7]:
df['hr_sin']=np.sin(2*np.pi*(df['hr']/24))
df['hr_cos']=np.cos(2*np.pi*(df['hr']/24))
df['mnth_sin']=np.sin(2*np.pi*(df['mnth']/12))
df['mnth_cos']=np.cos(2*np.pi*(df['mnth']/12))
df['week_sin']=np.sin(2*np.pi*(df['weekday']/7))
df['week_cos']=np.cos(2*np.pi*(df['weekday']/7))

Commute features : 

In [8]:
df['Rush_Hour']=((df['workingday']==1)&(df['hr'].isin([7,8,9,17,18,19]))).astype(bool)
df['Off_Peak']=(df['hr'].isin([0,1,2,3,4,5])).astype(bool)

In [9]:
df['Previous_hour_cnt']=df['cnt'].shift(1)
df['Previous_day_cnt']=df['cnt'].shift(24)

We create now weather based columns : 

In [16]:
df['delta_temp']=df['temp']-df['atemp']
df['temp*hum']=df['temp']*df['hum']
df['bad_weather']=(df['weathersit']>=3).astype(bool)
df['windspeed*temp']=(df['temp']*df['windspeed'])

In [11]:
X=df.drop(['cnt','casual','registered'],axis=1)
y=df['cnt']
X_train=X[:13903]
X_test=X[13903:]
y_train=y[:13903]
y_test=y[13903:]

In [18]:
from sklearn.ensemble import RandomForestRegressor 
v0=RandomForestRegressor()
v0.fit(X_train,y_train)

DTypePromotionError: The DType <class 'numpy.dtypes.DateTime64DType'> could not be promoted by <class 'numpy.dtypes.Float64DType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class

In [ ]:
from sklearn.metrics import mean_absolute_error 
y_pred_0=v0.predict(X_test) 
mae_0=mean_absolute_error(y_test,y_pred_0 )
(mae_0 /y_test.median())*100

Our first model is performing very well let's try to improve things further with RandomizedSearchCV 

In [ ]:
from sklearn.model_selection import RandomizedSearchCV 
param_grid={'n_estimators':[300,500,700],
           'min_samples_split':[2,6,8,10],
           'min_samples_leaf':[1,5,10,20],
           'max_features':['log2','sqrt',0.5,0.8]}
randomized_search=RandomizedSearchCV(estimator=v0,cv=5,n_iter=60,param_distributions=param_grid,n_jobs=-1,random_state=42)
randomized_search.fit(X_train,y_train) 
print(randomized_search.best_params_)

In [ ]:
v1=RandomForestRegressor(**randomized_search.best_params_,random_state=42,n_jobs=-1)
v1.fit(X_train,y_train)

In [ ]:
y_pred_1=v1.predict(X_test)
mae_1=mean_absolute_error(y_test,y_pred_1)
(mae_1/y_test.median())*100

In [ ]:
def extract_features(model):
    feature_names=model.feature_names_in_ 
    importances=pd.DataFrame({'Feature':feature_names,
                             'Importance':model.feature_importances_}).sort_values('Importance',ascending=False).reset_index(drop=True)
    return importances 

In [ ]:
extract_features(v0)